# Supply Chain Analytics

This notebook analyzes the integrated supply chain, holiday, and weather dataset
created during the data engineering stage.

The analysis focuses on demand patterns, inventory risk, promotions, supplier
lead time, and external factors such as holidays and weather.

The goal is to use PySpark aggregations and derived business metrics to identify
inventory-management insights.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("CS675SupplyChainAnalysis")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.1.2


In [2]:
data_path = "/home/jovyan/work/CS675_Supply_Chain_Project/data/processed/final_joined"

df = spark.read.parquet(data_path)

print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 91250
Columns: 25


## Research Question 1: How does demand differ between holidays and non-holidays?

This analysis compares average units sold, average demand forecast, and average
inventory level between federal holidays and non-holiday dates.

Average values are emphasized rather than total sales because the dataset contains
far more non-holiday dates than holiday dates. Comparing totals would mainly reflect
the difference in the number of records rather than differences in daily demand.

In [3]:
holiday_demand_df = (
    df
    .groupBy("Is_Holiday")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(F.avg("Demand_Forecast"), 2).alias("Avg_Demand_Forecast"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level")
    )
    .orderBy("Is_Holiday")
)

holiday_demand_df.show()

+----------+-------+--------------+-------------------+-------------------+
|Is_Holiday|Records|Avg_Units_Sold|Avg_Demand_Forecast|Avg_Inventory_Level|
+----------+-------+--------------+-------------------+-------------------+
|         0|  88500|         20.09|              20.12|             470.67|
|         1|   2750|         18.87|              18.88|              498.8|
+----------+-------+--------------+-------------------+-------------------+



In [4]:
holiday_avg = (
    df.filter(F.col("Is_Holiday") == 1)
    .agg(F.avg("Units_Sold"))
    .first()[0]
)

nonholiday_avg = (
    df.filter(F.col("Is_Holiday") == 0)
    .agg(F.avg("Units_Sold"))
    .first()[0]
)

holiday_difference_pct = (
    (holiday_avg - nonholiday_avg) / nonholiday_avg
) * 100

print("Holiday average:", round(holiday_avg, 2))
print("Non-holiday average:", round(nonholiday_avg, 2))
print("Percentage difference:", round(holiday_difference_pct, 2), "%")

Holiday average: 18.87
Non-holiday average: 20.09
Percentage difference: -6.07 %


### Finding

Average units sold were 18.87 on federal holidays compared with 20.09 on
non-holiday dates. Holiday demand was therefore approximately 6.07% lower
than non-holiday demand in this dataset.

At the same time, average inventory levels were higher on holidays
(498.80 units) than on non-holidays (470.67 units).

This may suggest that inventory levels were relatively high during holiday
periods despite lower observed demand. However, this analysis shows an
association only and does not establish that holidays caused the difference.

## Research Question 2: Does demand vary across individual federal holidays?

Treating all federal holidays as a single category may hide important differences
between specific holiday dates. This analysis compares average units sold,
inventory levels, demand forecasts, and promotion activity across individual
federal holidays.

In [5]:
individual_holiday_df = (
    df
    .filter(F.col("Is_Holiday") == 1)
    .groupBy("official_name")
    .agg(
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level"),
        F.round(F.avg("Demand_Forecast"), 2).alias("Avg_Demand_Forecast"),
        F.round(F.avg("Promotion_Flag") * 100, 2).alias("Promotion_Rate_Pct")
    )
    .orderBy(F.desc("Avg_Units_Sold"))
)

individual_holiday_df.show(truncate=False)

+------------------------------------+--------------+-------------------+-------------------+------------------+
|official_name                       |Avg_Units_Sold|Avg_Inventory_Level|Avg_Demand_Forecast|Promotion_Rate_Pct|
+------------------------------------+--------------+-------------------+-------------------+------------------+
|Washington's Birthday               |28.31         |463.41             |28.11              |9.6               |
|Memorial Day                        |26.39         |460.79             |26.38              |14.0              |
|Birthday of Martin Luther King, Jr. |22.75         |512.66             |22.96              |12.4              |
|Juneteenth National Independence Day|21.87         |453.0              |21.82              |6.8               |
|New Year's Day                      |20.22         |721.32             |20.2               |8.8               |
|Independence Day                    |19.64         |471.72             |19.46              |12.

### Finding

Demand varied substantially across individual federal holidays.

Washington's Birthday had the highest average units sold at 28.31, followed by
Memorial Day at 26.39. In contrast, Labor Day and Columbus Day had much lower
average demand at 10.88 and 10.68 units sold, respectively.

This shows that grouping all holidays together can hide meaningful differences
between specific holiday dates.

Promotion activity also varied by holiday. For example, Memorial Day and
Columbus Day both had promotion rates of 14%, while Juneteenth had a lower
promotion rate of 6.8%.

Because each holiday represents a single calendar date in 2024, these results
should be interpreted as descriptive patterns rather than evidence that a
holiday caused the observed demand level.

## Research Question 3: Are inventory-risk conditions more common during holidays?

The original Stockout_Flag contains only zero values, so it cannot distinguish
inventory risk.

A derived inventory-risk indicator is therefore created when Inventory_Level is
less than or equal to Reorder_Point. This represents a condition where inventory
has reached the level at which replenishment should be considered.

In [6]:
analysis_df = df.withColumn(
    "Inventory_Risk",
    F.when(
        F.col("Inventory_Level") <= F.col("Reorder_Point"),
        1
    ).otherwise(0)
)

In [7]:
analysis_df.groupBy("Inventory_Risk") \
    .count() \
    .orderBy("Inventory_Risk") \
    .show()

+--------------+-----+
|Inventory_Risk|count|
+--------------+-----+
|             0|86209|
|             1| 5041|
+--------------+-----+



In [8]:
holiday_risk_df = (
    analysis_df
    .groupBy("Is_Holiday")
    .agg(
        F.count("*").alias("Records"),
        F.sum("Inventory_Risk").alias("Risk_Records"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Is_Holiday")
)

holiday_risk_df.show()

+----------+-------+------------+-----------------------+
|Is_Holiday|Records|Risk_Records|Inventory_Risk_Rate_Pct|
+----------+-------+------------+-----------------------+
|         0|  88500|        4913|                   5.55|
|         1|   2750|         128|                   4.65|
+----------+-------+------------+-----------------------+



### Finding

A total of 5,041 records were identified as inventory-risk conditions, where
Inventory_Level was less than or equal to Reorder_Point.

The inventory-risk rate was 5.55% on non-holiday dates compared with 4.65% on
federal holidays.

This means that inventory-risk conditions were slightly less common on holidays
in this dataset.

Combined with the earlier finding that average inventory levels were higher on
holidays, this may suggest that inventory was relatively well positioned during
holiday periods. However, the analysis is descriptive and does not establish
that holidays caused the lower inventory-risk rate.

## Research Question 4: How do promotions affect demand and inventory risk?

Promotions may increase customer demand and create additional pressure on inventory.
This analysis compares average units sold, inventory levels, and inventory-risk
rates between promoted and non-promoted records.

In [9]:
promotion_analysis_df = (
    analysis_df
    .groupBy("Promotion_Flag")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level"),
        F.round(F.avg("Demand_Forecast"), 2).alias("Avg_Demand_Forecast"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Promotion_Flag")
)

promotion_analysis_df.show()

+--------------+-------+--------------+-------------------+-------------------+-----------------------+
|Promotion_Flag|Records|Avg_Units_Sold|Avg_Inventory_Level|Avg_Demand_Forecast|Inventory_Risk_Rate_Pct|
+--------------+-------+--------------+-------------------+-------------------+-----------------------+
|             0|  81980|          19.5|             472.12|              19.53|                   5.34|
|             1|   9270|         24.91|             466.26|              24.93|                    7.2|
+--------------+-------+--------------+-------------------+-------------------+-----------------------+



In [10]:
holiday_promotion_df = (
    analysis_df
    .groupBy("Is_Holiday", "Promotion_Flag")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Is_Holiday", "Promotion_Flag")
)

holiday_promotion_df.show()

+----------+--------------+-------+--------------+-----------------------+
|Is_Holiday|Promotion_Flag|Records|Avg_Units_Sold|Inventory_Risk_Rate_Pct|
+----------+--------------+-------+--------------+-----------------------+
|         0|             0|  79537|         19.54|                   5.36|
|         0|             1|   8963|         24.98|                   7.24|
|         1|             0|   2443|         18.35|                    4.5|
|         1|             1|    307|         23.04|                   5.86|
+----------+--------------+-------+--------------+-----------------------+



### Finding

Promotions were associated with higher demand and higher inventory risk.

Non-promoted records had an average of 19.50 units sold, while promoted records
had an average of 24.91 units sold. This is an increase of about 27.7% during
promotional periods.

At the same time, the inventory-risk rate increased from 5.34% for non-promoted
records to 7.20% for promoted records. Average inventory levels were also slightly
lower during promotions, at 466.26 units compared with 472.12 units for
non-promoted records.

The holiday comparison shows the same general pattern. On non-holiday dates,
promoted records averaged 24.98 units sold compared with 19.54 without promotions.
On holiday dates, promoted records averaged 23.04 units sold compared with 18.35
without promotions.

Inventory risk was also higher under promotion in both groups. The risk rate rose
from 5.36% to 7.24% on non-holidays and from 4.50% to 5.86% on holidays.

Overall, promotions in this dataset were associated with stronger demand but also
greater pressure on inventory. These results are descriptive and do not prove
that promotions directly caused the increase in demand or inventory risk.

## Research Question 5: How does supplier lead time relate to inventory risk?

Longer supplier lead times may make replenishment more difficult because inventory
must cover demand for a longer period before new stock arrives.

This analysis groups supplier lead times into shorter, medium, and longer lead-time
categories and compares their average demand, inventory level, and inventory-risk rate.

In [12]:
leadtime_df = analysis_df.withColumn(
    "Lead_Time_Category",
    F.when(F.col("Supplier_Lead_Time_Days") <= 5, "Short")
     .when(F.col("Supplier_Lead_Time_Days") <= 10, "Medium")
     .otherwise("Long")
)

In [13]:
leadtime_analysis_df = (
    leadtime_df
    .groupBy("Lead_Time_Category")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Supplier_Lead_Time_Days"), 2).alias("Avg_Lead_Time_Days"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Avg_Lead_Time_Days")
)

leadtime_analysis_df.show()

+------------------+-------+------------------+--------------+-------------------+-----------------------+
|Lead_Time_Category|Records|Avg_Lead_Time_Days|Avg_Units_Sold|Avg_Inventory_Level|Inventory_Risk_Rate_Pct|
+------------------+-------+------------------+--------------+-------------------+-----------------------+
|             Short|  29930|              3.38|         20.03|             470.99|                    5.5|
|            Medium|  32120|              8.07|         20.08|             469.48|                   5.56|
|              Long|  29200|             12.61|         20.05|             474.32|                   5.51|
+------------------+-------+------------------+--------------+-------------------+-----------------------+



In [14]:
high_demand_leadtime_df = (
    leadtime_df
    .filter(F.col("Units_Sold") >= 31)
    .groupBy("Lead_Time_Category")
    .agg(
        F.count("*").alias("High_Demand_Records"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Lead_Time_Category")
)

high_demand_leadtime_df.show()

+------------------+-------------------+-------------------+-----------------------+
|Lead_Time_Category|High_Demand_Records|Avg_Inventory_Level|Inventory_Risk_Rate_Pct|
+------------------+-------------------+-------------------+-----------------------+
|              Long|               3968|             455.67|                  10.06|
|            Medium|               4242|             452.43|                  10.54|
|             Short|               3993|             453.68|                  10.39|
+------------------+-------------------+-------------------+-----------------------+



### Finding

Supplier lead time showed very little relationship with overall inventory risk.

The short lead-time group averaged 3.38 days and had an inventory-risk rate of
5.50%. The medium group averaged 8.07 days with a risk rate of 5.56%, while the
long lead-time group averaged 12.61 days with a risk rate of 5.51%.

Average demand was also very similar across the three lead-time groups, at
approximately 20 units sold.

When the analysis was limited to high-demand records, inventory-risk rates
increased to approximately 10% across all three groups. However, the differences
between short, medium, and long lead times remained small:

- Long lead time: 10.06%
- Medium lead time: 10.54%
- Short lead time: 10.39%

This suggests that high demand was associated with substantially greater inventory
risk, but longer supplier lead time by itself did not show a strong relationship
with inventory risk in this dataset.

Because this is an observational analysis, the results describe associations and
do not establish a causal effect of supplier lead time on inventory risk.

## Research Question 6: How accurate is the demand forecast?

Forecast accuracy is important for inventory planning because large forecast
errors may contribute to excess inventory or inventory shortages.

This analysis measures the difference between actual units sold and the demand
forecast using forecast error and absolute forecast error.

Forecast Error = Units_Sold - Demand_Forecast

Absolute Forecast Error = |Units_Sold - Demand_Forecast|

The absolute error is useful because positive and negative errors cannot cancel
each other out.

In [17]:
forecast_df = (
    analysis_df
    .withColumn(
        "Forecast_Error",
        F.col("Units_Sold") - F.col("Demand_Forecast")
    )
    .withColumn(
        "Absolute_Forecast_Error",
        F.abs(F.col("Units_Sold") - F.col("Demand_Forecast"))
    )
)

In [18]:
overall_forecast_df = (
    forecast_df
    .agg(
        F.round(F.avg("Forecast_Error"), 2).alias("Mean_Forecast_Error"),
        F.round(F.avg("Absolute_Forecast_Error"), 2).alias("Mean_Absolute_Error"),
        F.round(F.max("Absolute_Forecast_Error"), 2).alias("Max_Absolute_Error")
    )
)

overall_forecast_df.show()

+-------------------+-------------------+------------------+
|Mean_Forecast_Error|Mean_Absolute_Error|Max_Absolute_Error|
+-------------------+-------------------+------------------+
|              -0.03|               2.38|             14.08|
+-------------------+-------------------+------------------+



In [19]:
holiday_forecast_df = (
    forecast_df
    .groupBy("Is_Holiday")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Forecast_Error"), 2).alias("Mean_Forecast_Error"),
        F.round(F.avg("Absolute_Forecast_Error"), 2).alias("Mean_Absolute_Error")
    )
    .orderBy("Is_Holiday")
)

holiday_forecast_df.show()

+----------+-------+-------------------+-------------------+
|Is_Holiday|Records|Mean_Forecast_Error|Mean_Absolute_Error|
+----------+-------+-------------------+-------------------+
|         0|  88500|              -0.03|               2.38|
|         1|   2750|              -0.01|               2.41|
+----------+-------+-------------------+-------------------+



In [20]:
promotion_forecast_df = (
    forecast_df
    .groupBy("Promotion_Flag")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Forecast_Error"), 2).alias("Mean_Forecast_Error"),
        F.round(F.avg("Absolute_Forecast_Error"), 2).alias("Mean_Absolute_Error")
    )
    .orderBy("Promotion_Flag")
)

promotion_forecast_df.show()


+--------------+-------+-------------------+-------------------+
|Promotion_Flag|Records|Mean_Forecast_Error|Mean_Absolute_Error|
+--------------+-------+-------------------+-------------------+
|             0|  81980|              -0.03|               2.38|
|             1|   9270|              -0.01|               2.36|
+--------------+-------+-------------------+-------------------+



### Finding

The demand forecast performed consistently well across the dataset.

The overall mean forecast error was -0.03 units, which is very close to zero.
This indicates that the forecasts were nearly unbiased on average, with only a
slight tendency to overforecast actual demand.

The mean absolute error was 2.38 units, meaning that the forecast differed from
actual units sold by about 2.38 units on average. The maximum absolute error was
14.08 units.

Forecast accuracy was also very similar between holiday and non-holiday records.
The mean absolute error was 2.38 units on non-holidays and 2.41 units on holidays.

Promotions also did not substantially reduce forecast accuracy. The mean absolute
error was 2.38 units for non-promoted records and 2.36 units for promoted records.

Overall, the forecasting process remained relatively stable across holiday and
promotion conditions. Although the average forecast error was close to zero,
absolute error was also evaluated so that positive and negative forecast errors
would not cancel each other out.

## Research Question 7: What is the estimated financial impact of demand patterns?

Supply chain decisions affect not only inventory availability but also revenue
and gross profit.

This analysis estimates revenue and gross profit using the available unit price
and unit cost fields.

Estimated Revenue = Units_Sold × Unit_Price

Estimated Gross Profit = Units_Sold × (Unit_Price - Unit_Cost)

These are simplified estimates based on the dataset and do not include other
operating costs, freight, taxes, discounts, or overhead.

In [21]:
finance_df = (
    forecast_df
    .withColumn(
        "Estimated_Revenue",
        F.col("Units_Sold") * F.col("Unit_Price")
    )
    .withColumn(
        "Estimated_Gross_Profit",
        F.col("Units_Sold") *
        (F.col("Unit_Price") - F.col("Unit_Cost"))
    )
)

In [22]:
overall_finance_df = (
    finance_df
    .agg(
        F.round(F.sum("Estimated_Revenue"), 2).alias("Estimated_Total_Revenue"),
        F.round(F.sum("Estimated_Gross_Profit"), 2).alias("Estimated_Total_Gross_Profit"),
        F.round(F.avg("Estimated_Revenue"), 2).alias("Avg_Revenue_Per_Record"),
        F.round(F.avg("Estimated_Gross_Profit"), 2).alias("Avg_Gross_Profit_Per_Record")
    )
)

overall_finance_df.show(truncate=False)

+-----------------------+----------------------------+----------------------+---------------------------+
|Estimated_Total_Revenue|Estimated_Total_Gross_Profit|Avg_Revenue_Per_Record|Avg_Gross_Profit_Per_Record|
+-----------------------+----------------------------+----------------------+---------------------------+
|3.342633722E7          |1.108820123E7               |366.32                |121.51                     |
+-----------------------+----------------------------+----------------------+---------------------------+



In [23]:
holiday_finance_df = (
    finance_df
    .groupBy("Is_Holiday")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Estimated_Revenue"), 2).alias("Avg_Revenue_Per_Record"),
        F.round(F.avg("Estimated_Gross_Profit"), 2).alias("Avg_Gross_Profit_Per_Record")
    )
    .orderBy("Is_Holiday")
)

holiday_finance_df.show()

+----------+-------+----------------------+---------------------------+
|Is_Holiday|Records|Avg_Revenue_Per_Record|Avg_Gross_Profit_Per_Record|
+----------+-------+----------------------+---------------------------+
|         0|  88500|                366.99|                     121.74|
|         1|   2750|                344.51|                     114.17|
+----------+-------+----------------------+---------------------------+



In [24]:
promotion_finance_df = (
    finance_df
    .groupBy("Promotion_Flag")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Estimated_Revenue"), 2).alias("Avg_Revenue_Per_Record"),
        F.round(F.avg("Estimated_Gross_Profit"), 2).alias("Avg_Gross_Profit_Per_Record"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold")
    )
    .orderBy("Promotion_Flag")
)

promotion_finance_df.show()

+--------------+-------+----------------------+---------------------------+--------------+
|Promotion_Flag|Records|Avg_Revenue_Per_Record|Avg_Gross_Profit_Per_Record|Avg_Units_Sold|
+--------------+-------+----------------------+---------------------------+--------------+
|             0|  81980|                356.08|                      118.1|          19.5|
|             1|   9270|                456.85|                     151.68|         24.91|
+--------------+-------+----------------------+---------------------------+--------------+



### Finding

The dataset generated approximately $33.43 million in estimated revenue and
$11.09 million in estimated gross profit.

Average estimated revenue was $366.32 per record, while average estimated gross
profit was $121.51 per record.

Holiday records produced slightly lower average financial results than
non-holiday records. Average revenue per record was $344.51 on holidays compared
with $366.99 on non-holidays. Average gross profit per record was $114.17 on
holidays compared with $121.74 on non-holidays.

Promotions showed a much stronger financial effect. Promoted records averaged
$456.85 in estimated revenue and $151.68 in estimated gross profit, compared with
$356.08 in revenue and $118.10 in gross profit for non-promoted records.

This result is consistent with the earlier finding that promotions were associated
with higher unit sales. However, promotions were also associated with higher
inventory risk, so the higher estimated financial return may come with greater
replenishment pressure.

These financial measures are simplified estimates based on Units_Sold, Unit_Price,
and Unit_Cost. They do not include freight, discounts, overhead, taxes, or other
operating costs.

## Research Question 8: Are daily weather conditions associated with demand or inventory risk?

Weather may influence transportation, customer demand, or supply chain operations.

However, the available weather dataset comes from a single station in Missouri,
while the supply chain dataset does not provide specific warehouse geographic
locations.

Therefore, this analysis treats weather only as a general daily external factor.
The results should not be interpreted as location-specific weather effects.

In [25]:
weather_analysis_df = (
    analysis_df
    .filter(F.col("TEMP").isNotNull())
)

In [26]:
weather_analysis_df = weather_analysis_df.withColumn(
    "Temperature_Category",
    F.when(F.col("TEMP") < 32, "Cold")
     .when(F.col("TEMP") < 70, "Moderate")
     .otherwise("Warm")
)

In [27]:
temperature_analysis_df = (
    weather_analysis_df
    .groupBy("Temperature_Category")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("TEMP"), 2).alias("Avg_Temperature"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Avg_Temperature")
)

temperature_analysis_df.show()

+--------------------+-------+---------------+--------------+-------------------+-----------------------+
|Temperature_Category|Records|Avg_Temperature|Avg_Units_Sold|Avg_Inventory_Level|Inventory_Risk_Rate_Pct|
+--------------------+-------+---------------+--------------+-------------------+-----------------------+
|                Cold|   4250|          21.23|         22.18|             505.29|                   4.14|
|            Moderate|  48750|          53.24|         22.63|             470.85|                    6.2|
|                Warm|  23250|          76.34|         20.07|             465.29|                    5.9|
+--------------------+-------+---------------+--------------+-------------------+-----------------------+



In [28]:
precipitation_df = (
    weather_analysis_df
    .filter(F.col("PRCP").isNotNull())
    .withColumn(
        "Has_Precipitation",
        F.when(F.col("PRCP") > 0, 1).otherwise(0)
    )
)

precipitation_analysis_df = (
    precipitation_df
    .groupBy("Has_Precipitation")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Has_Precipitation")
)

precipitation_analysis_df.show()


+-----------------+-------+--------------+-------------------+-----------------------+
|Has_Precipitation|Records|Avg_Units_Sold|Avg_Inventory_Level|Inventory_Risk_Rate_Pct|
+-----------------+-------+--------------+-------------------+-----------------------+
|                0|  42250|         21.55|             473.19|                   5.95|
+-----------------+-------+--------------+-------------------+-----------------------+



In [30]:
weather_analysis_df.select("PRCP") \
    .distinct() \
    .orderBy("PRCP") \
    .show(50)

+----+
|PRCP|
+----+
|NULL|
| 0.0|
+----+



In [31]:
weather_analysis_df.groupBy("PRCP") \
    .count() \
    .orderBy("PRCP") \
    .show(50)

+----+-----+
|PRCP|count|
+----+-----+
|NULL|34000|
| 0.0|42250|
+----+-----+



In [32]:
print(
    "Positive precipitation rows:",
    weather_analysis_df.filter(F.col("PRCP") > 0).count()
)

Positive precipitation rows: 0


In [29]:
# Compare days with and without measurable precipitation.

precipitation_df = (
    weather_analysis_df
    .filter(F.col("PRCP").isNotNull())
    .withColumn(
        "Has_Precipitation",
        F.when(F.col("PRCP") > 0, 1).otherwise(0)
    )
)

precipitation_analysis_df = (
    precipitation_df
    .groupBy("Has_Precipitation")
    .agg(
        F.count("*").alias("Records"),
        F.round(F.avg("PRCP"), 2).alias("Avg_Precipitation"),
        F.round(F.avg("Units_Sold"), 2).alias("Avg_Units_Sold"),
        F.round(F.avg("Inventory_Level"), 2).alias("Avg_Inventory_Level"),
        F.round(
            F.avg("Inventory_Risk") * 100,
            2
        ).alias("Inventory_Risk_Rate_Pct")
    )
    .orderBy("Has_Precipitation")
)

precipitation_analysis_df.show()

+-----------------+-------+-----------------+--------------+-------------------+-----------------------+
|Has_Precipitation|Records|Avg_Precipitation|Avg_Units_Sold|Avg_Inventory_Level|Inventory_Risk_Rate_Pct|
+-----------------+-------+-----------------+--------------+-------------------+-----------------------+
|                0|  42250|              0.0|         21.55|             473.19|                   5.95|
+-----------------+-------+-----------------+--------------+-------------------+-----------------------+



### Finding

Temperature categories showed noticeable differences in demand, inventory levels,
and inventory risk.

Cold days had an average temperature of 21.23°F, average units sold of 22.18,
and the highest average inventory level at 505.29 units. Their inventory-risk
rate was the lowest at 4.14%.

Moderate days averaged 53.24°F and had the highest average demand at 22.63 units
sold. However, they also had the highest inventory-risk rate at 6.20%.

Warm days averaged 76.34°F and showed lower average demand at 20.07 units sold.
Their average inventory level was 465.29 units, and the inventory-risk rate was
5.90%.

These results suggest that weather conditions may coincide with differences in
demand and inventory behavior. However, the weather data comes from a single
Missouri station and cannot be matched to specific warehouse locations, so the
results should be treated as exploratory rather than location-specific effects.

### Precipitation Limitation

After cleaning the weather data, there were no positive precipitation observations
remaining in the usable dataset.

All non-null precipitation values were equal to 0.0, so there was no meaningful
variation available to compare precipitation days with non-precipitation days.

Therefore, precipitation was excluded from the final weather interpretation.
The weather analysis focuses on temperature categories, where sufficient variation
was available for comparison.

# Overall Findings

The integrated analysis identified several supply chain and business patterns
across the 2024 inventory dataset.

1. **Holiday demand was slightly lower overall.**
   Average units sold were 18.87 on federal holidays compared with 20.09 on
   non-holiday dates, a decrease of approximately 6.07%.

2. **Individual holidays behaved differently.**
   Washington's Birthday and Memorial Day showed relatively high average demand,
   while Labor Day and Columbus Day showed lower demand. This indicates that a
   single holiday indicator can hide important differences between specific dates.

3. **Inventory risk was relatively low overall.**
   The derived inventory-risk rate was 5.55% on non-holidays and 4.65% on holidays.
   Holiday inventory levels were also higher on average.

4. **Promotions had one of the strongest relationships with demand.**
   Promoted records averaged 24.91 units sold compared with 19.50 units without
   promotions. However, inventory-risk rates also increased from 5.34% to 7.20%
   during promotional periods.

5. **Supplier lead time showed little direct relationship with inventory risk.**
   Risk rates were very similar across short, medium, and long lead-time groups.
   High-demand periods showed a much stronger relationship with inventory risk,
   with risk rates rising to approximately 10%.

6. **Demand forecasts were relatively accurate and stable.**
   The mean forecast error was -0.03 units and mean absolute error was 2.38 units.
   Forecast accuracy remained similar across holiday and promotional conditions.

7. **Promotions were also associated with stronger financial performance.**
   The dataset produced approximately $33.43 million in estimated revenue and
   $11.09 million in estimated gross profit. Promoted records had higher average
   estimated revenue and gross profit than non-promoted records, but they also
   carried greater inventory risk.

8. **Weather analysis was exploratory.**
   Temperature categories showed differences in demand and inventory behavior,
   but the weather source represented only one Missouri station and could not be
   geographically matched to individual warehouses. Precipitation was excluded
   from final interpretation because no positive precipitation observations
   remained after cleaning.

## Business Interpretation

The results suggest that demand intensity and promotional activity are more
important indicators of inventory pressure than holidays or supplier lead-time
categories alone.

Promotions may generate higher sales and estimated gross profit, but they also
increase the likelihood that inventory reaches its reorder threshold. Therefore,
promotion planning should be coordinated with inventory availability and
replenishment decisions.

The analysis also demonstrates the value of integrating multiple data sources.
Holiday and weather data added external context to the inventory fact table,
while PySpark enabled the datasets to be cleaned, joined, transformed, and
analyzed in a reproducible workflow.

## Limitations

- The supply chain dataset covers only one year.
- Each federal holiday represents only one date in 2024.
- The original Stockout_Flag contained no positive values, so inventory risk was
  derived using Inventory_Level <= Reorder_Point.
- The weather dataset represents a single station and has incomplete date coverage.
- Revenue and gross profit are simplified estimates and exclude freight,
  discounts, overhead, taxes, and other business costs.
- All results are descriptive associations and should not be interpreted as
  causal relationships.